# Step 6. 산불발생 인간활동 원인프록시 분석

발생원인 결측 감사와 등산로·임도·생활권 접근성 프록시 분석에 필요한 전체 자료를 준비한다.

## 현재 구현 범위

이 노트북은 **원천 데이터 경로 확인, 로딩, 필수 스키마 검증, 시간 파싱,
기본 행 수 감사**까지 구현한다. 통계 분석과 시각화는 이후 셀에서 이어서 작성한다.

공통 데이터 계약은 `README.md`를 따른다. Step 2~6에서 캐나다 지수를 사건 시각에
결합할 때는 12시 이전이면 전일 정오, 12시 이후이면 당일 정오 자료만 사용한다.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
candidates = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
REPO_ROOT = next(
    (path for path in candidates if (path / "jsw/강원_재_EDA/re_eda_common.py").exists()),
    Path(r"D:/farm-system-public-02"),
)
MODULE_DIR = REPO_ROOT / "jsw/강원_재_EDA"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from re_eda_common import (
    DATA_PATHS,
    DERIVED_WEATHER_COLUMNS,
    RAW_WEATHER_COLUMNS,
    add_canadian_asof_keys,
    check_sources,
    configure_notebook,
    frame_inventory,
    load_access_lines,
    load_canadian_indices,
    load_dem_metadata,
    load_fire,
    load_grid_bundle,
    load_hourly_weather,
    load_infrastructure,
    load_landcover,
    load_roads,
    load_terrain,
)

configure_notebook()

## 1. 원천 파일 존재 여부

In [ ]:
SOURCE_KEYS = ['fire', 'weather_cells', 'weather_grid', 'weather_hourly_derived', 'climate_type', 'canada_ffmc', 'canada_fwi', 'terrain', 'dem', 'landcover', 'roads', 'trails', 'forest_roads', 'fire_stations', 'fire_water']
source_audit = check_sources(SOURCE_KEYS)
display(source_audit)

## 2. 산불, 격자, 지형 자료 로딩

In [ ]:
fire, fire_points = load_fire()
grid_bundle = load_grid_bundle()
weather_cells = grid_bundle["cells"]
climate_type = grid_bundle["climate"]
weather_grid = grid_bundle["grid"]
terrain = load_terrain()
dem_metadata = load_dem_metadata()

print("산불 포인트 CRS:", fire_points.crs)
print("기상 격자 CRS:", weather_grid.crs)
display(terrain.head())
display(dem_metadata)

## 3. 대용량 공간 원천 로딩

토지피복 약 1.2GB, 도로 약 307MB 파일을 실제로 읽는다.
메모리가 부족한 환경에서는 두 줄을 각각 실행해 사용량을 확인한다.

In [ ]:
landcover = load_landcover()
roads = load_roads()

print("토지피복:", len(landcover), "건 / CRS:", landcover.crs)
print("도로:", len(roads), "건 / CRS:", roads.crs)

## 4. 등산로, 임도, 소방 인프라 로딩

In [ ]:
access_lines = load_access_lines()
trails = access_lines["trails"]
forest_roads = access_lines["forest_roads"]

infrastructure = load_infrastructure()
fire_stations = infrastructure["fire_stations"]
fire_water = infrastructure["fire_water"]

print("등산로:", len(trails), "건")
print("임도:", len(forest_roads), "건")
print("소방서:", len(fire_stations), "건")
print("소방용수:", len(fire_water), "건")

## 5. 누수 방지 기상 및 캐나다 지수 로딩

In [ ]:
hourly_weather = load_hourly_weather(
    derived=True,
    columns=DERIVED_WEATHER_COLUMNS,
)
canadian_indices = load_canadian_indices()
fire = add_canadian_asof_keys(fire, time_column="기준시각")
assert (fire["캐나다지수_기준시각"] <= fire["기준시각"]).all()

## 6. 원인·피해·진화시간 결측 감사

In [ ]:
audit_columns = ["발생원인명", "피해면적(ha)", "피해금액", "진화소요시간(HH)"]
missing_audit = (
    fire[audit_columns]
    .isna()
    .agg(["sum", "mean"])
    .T
    .rename(columns={"sum": "결측건수", "mean": "결측률"})
)
display(missing_audit)

## 로딩 결과 요약

In [ ]:
loaded_frames = {
    "fire": fire,
    "fire_points": fire_points,
    "weather_cells": weather_cells,
    "climate_type": climate_type,
    "weather_grid": weather_grid,
    "terrain": terrain,
    "landcover": landcover,
    "roads": roads,
    "trails": trails,
    "forest_roads": forest_roads,
    "fire_stations": fire_stations,
    "fire_water": fire_water,
    "hourly_weather": hourly_weather,
    "canadian_indices": canadian_indices,
}
display(frame_inventory(loaded_frames))

## 다음 구현 범위

공간 최단거리를 계산해 250m·500m·1,000m 입산활동 프록시와 생활권 프록시를 만들고 매칭 비발생 및 선형 공간 대조군과 비교한다.

현재 노트북은 로딩과 입력 감사까지만 실행한다. 이후 분석 셀에서도
`README.md`의 미래 정보 누수 방지 규칙과 대조군 정의를 유지해야 한다.